# RQ1 - Effectiveness and Baselines

This notebook reconstructs the RQ1 tables and visualizations used in the article from the released CSV files. The generated figures are written to `figures/reproduced/`; the exact article figures are displayed after each reconstruction.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
REPRODUCED = FIGURES / 'reproduced'
REPRODUCED.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save_reproduced(fig, name):
    fig.savefig(REPRODUCED / f'{name}.png', bbox_inches='tight')
    fig.savefig(REPRODUCED / f'{name}.pdf', bbox_inches='tight')

def pm(mean, std, digits=2):
    return f'{mean:.{digits}f} ± {std:.{digits}f}'

def show_article_figure(filename):
    display(Markdown(f'**Article figure:** `figures/{filename}`'))
    display(Image(filename=str(FIGURES / filename)))

from matplotlib.colors import LinearSegmentedColormap

RQ1 = RESULTS / 'rq1_effectiveness_baselines'
PALETTE = {
    'MetaMatch': '#1F77B4',
    'LLMATCH': '#D62728',
    'SMUTF': '#9467BD',
    'MagnetoFTGPT': '#FF7F0E',
    'MagnetoFT': '#FDB462',
    'MagnetoGPT': '#E377C2',
    'Magneto': '#8C564B',
    'ISResMat': '#17BECF',
    'COMA++': '#7F7F7F',
    'COMA': '#BCBD22',
    'Similarity Flooding': '#2CA02C',
    'Distribution Based': '#AEC7E8',
    'Cupid': '#C5B0D5',
}
ORDER = list(PALETTE)


## Table 1 - MetaMatch Classifier Comparison

In [ ]:
classifiers = pd.read_csv(RQ1 / 'table_1_metamatch_classifiers_all_to_all.csv')
classifier_order = [
    'MetaMatch Random Forest', 'MetaMatch XGBoost', 'MetaMatch CatBoost',
    'MetaMatch SVM', 'MetaMatch Logistic Regression'
]
classifiers['classifier'] = pd.Categorical(classifiers['classifier'], classifier_order, ordered=True)
classifiers = classifiers.sort_values('classifier')

paper_classifier_table = pd.DataFrame({
    'Classifier': classifiers['classifier'].astype(str).str.replace('MetaMatch ', '', regex=False),
    'F1': [pm(m, s) for m, s in zip(classifiers['mean_f1_all_to_all'], classifiers['std_f1_all_to_all'])],
    'Precision': [pm(m, s) for m, s in zip(classifiers['mean_precision_all_to_all'], classifiers['std_precision_all_to_all'])],
    'Recall': [pm(m, s) for m, s in zip(classifiers['mean_recall_all_to_all'], classifiers['std_recall_all_to_all'])],
})
display(paper_classifier_table)
print(paper_classifier_table.to_latex(index=False, escape=False))

## Table 2 - Effectiveness Against Baselines

In [ ]:
baselines = pd.read_csv(RQ1 / 'table_1_effectiveness_baselines_summary.csv')
baselines = baselines[baselines['method'].isin(ORDER)].copy()
baselines['method'] = pd.Categorical(baselines['method'], ORDER, ordered=True)
baselines = baselines.sort_values('method')

paper_baseline_table = pd.DataFrame({
    'Method': baselines['method'].astype(str),
    'F1': [pm(m, s) for m, s in zip(baselines['mean_f1'], baselines['std_f1'])],
    'Precision': [pm(m, s) for m, s in zip(baselines['mean_precision'], baselines['std_precision'])],
    'Recall': [pm(m, s) for m, s in zip(baselines['mean_recall'], baselines['std_recall'])],
})
display(paper_baseline_table)
print(paper_baseline_table.to_latex(index=False, escape=False))

## Table 3 - Paired Statistical Tests

In [ ]:
stats = pd.read_csv(RQ1 / 'rq1_metamatch_vs_baselines_paired_stat_tests_paper.csv')
stats['baseline'] = pd.Categorical(stats['baseline'], [m for m in ORDER if m != 'MetaMatch'], ordered=True)
stats = stats.sort_values('baseline')

def p_fmt(x):
    return '<0.001' if x < 0.001 else f'{x:.3f}'

paper_stats_table = pd.DataFrame({
    'Baseline': stats['baseline'].astype(str),
    'ΔF1': stats['mean_f1_diff'].map(lambda x: f'{x:.2f}'),
    'Holm p-value': stats['holm_p'].map(p_fmt),
    'Significant': stats['significant_holm_0_05'].map(lambda x: 'Yes' if bool(x) else 'No'),
})
display(paper_stats_table)
print(paper_stats_table.to_latex(index=False, escape=False))

## Figure 2 Top - F1 Distribution

In [ ]:
pair_scores = pd.read_csv(RQ1 / 'figure_2_effectiveness_baselines_metamatch_by_pair.csv')
pair_scores = pair_scores[pair_scores['method'].isin(ORDER)].copy()
fig, ax = plt.subplots(figsize=(10, 4.8))
positions = np.arange(len(ORDER))
data = [pair_scores.loc[pair_scores['method'] == m, 'f1_all_to_all'].dropna().to_numpy() for m in ORDER]
parts = ax.violinplot(data, positions=positions, showmeans=True, showmedians=False, widths=0.8)
for body, method in zip(parts['bodies'], ORDER):
    body.set_facecolor(PALETTE[method])
    body.set_edgecolor('#222222')
    body.set_alpha(0.75)
for key in ['cmins', 'cmaxes', 'cbars', 'cmeans']:
    parts[key].set_color('#222222')
    parts[key].set_linewidth(1.0)
ax.set_xticks(positions)
ax.set_xticklabels(ORDER, rotation=45, ha='right')
ax.set_ylabel('F1')
ax.set_ylim(-0.02, 1.02)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
save_reproduced(fig, 'plot_distribution_baseline')
plt.show()
show_article_figure('plot_distribution_baseline.png')

## Figure 2 Bottom - Pairwise Win-rate Heatmap

In [ ]:
win = pd.read_csv(RQ1 / 'figure_2_winrate_f1_all_to_all_no_ties_percent_matrix.csv', index_col=0)
methods = [m for m in ORDER if m in win.index and m in win.columns]
win = win.loc[methods, methods].astype(float).round(2)
np.fill_diagonal(win.values, np.nan)

cmap = LinearSegmentedColormap.from_list('red_white_green', ['#C92A2A', '#FFFFFF', '#2B8A3E'])
fig, ax = plt.subplots(figsize=(10.5, 8.5))
im = ax.imshow(np.ma.masked_invalid(win.to_numpy()), cmap=cmap, vmin=0, vmax=100)
ax.set_xticks(np.arange(len(methods)))
ax.set_yticks(np.arange(len(methods)))
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.set_yticklabels(methods)
for i in range(len(methods)):
    for j in range(len(methods)):
        value = win.iloc[i, j]
        label = '-' if pd.isna(value) else f'{value:.2f}'
        color = 'white' if not pd.isna(value) and (value < 30 or value > 70) else '#1f2937'
        ax.text(j, i, label, ha='center', va='center', fontsize=7.5, color=color)
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
fig.tight_layout()
save_reproduced(fig, 'plot_winrate_f1_all_to_all_no_ties_matplotlib_percent_heatmap')
plt.show()
show_article_figure('plot_winrate_f1_all_to_all_no_ties_matplotlib_percent_heatmap.png')

## Figure 3 - Breakdowns by Relation and Dataset

In [ ]:
group_scores = pd.read_csv(RQ1 / 'figure_rq1_effectiveness_by_group_pair_values.csv')
group_scores = group_scores[group_scores['method'].isin(ORDER)].copy()

def grouped_boxplot(df, group_col, filename):
    groups = sorted(df[group_col].dropna().unique())
    fig, axes = plt.subplots(len(groups), 1, figsize=(10, 2.5 * len(groups)), sharex=True)
    if len(groups) == 1:
        axes = [axes]
    present = ORDER
    for ax, group in zip(axes, groups):
        sub = df[df[group_col] == group]
        data = [sub.loc[sub['method'] == m, 'f1_all_to_all'].dropna().to_numpy() for m in present]
        bp = ax.boxplot(data, patch_artist=True, widths=0.65, showfliers=False)
        for patch, method in zip(bp['boxes'], present):
            patch.set_facecolor(PALETTE[method])
            patch.set_alpha(0.75)
        ax.set_ylabel(str(group))
        ax.set_ylim(-0.02, 1.02)
        ax.grid(axis='y', alpha=0.25)
    axes[-1].set_xticks(range(1, len(present) + 1))
    axes[-1].set_xticklabels(present, rotation=45, ha='right')
    fig.tight_layout()
    save_reproduced(fig, filename)
    plt.show()

grouped_boxplot(group_scores, 'relation_group', 'plot_effectiveness_by_relation_distribution')
show_article_figure('plot_effectiveness_by_relation_distribution.png')
grouped_boxplot(group_scores, 'dataset_group', 'plot_effectiveness_by_dataset_distribution')
show_article_figure('plot_effectiveness_by_dataset_distribution.png')